# Estrategia Multi-Cloud: Migración, Costos y Gobierno Avanzado

## Objetivos
- Decidir cuándo multi-cloud aporta valor vs complejidad innecesaria.
- Definir marco de evaluación (estratégico, financiero, técnico, riesgo).
- Diseñar plan de migración por fases con control de costos y riesgos.
- Implementar simulador de asignación dinámica de workloads.
- Formalizar decisión con ADR y métricas de éxito (KPIs).

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Gobernanza Semántica y Knowledge Graphs para Discovery y Reutilización (Estratégico)](06_gobernanza_semantica_y_kg_descubrimiento_reutilizacion.ipynb) | Siguiente: [Visualización Ejecutiva y Análisis de Impacto (Estratégico)](08_visualizacion_ejecutiva_analisis_impacto.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. Objetivos
2. 1. Framework de Decisión Single vs Multi-Cloud
3. 2. Modelo de Migración por Fases
4. 3. Simulador de Asignación Dinámica de Workloads

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Estrategia Multi-Cloud: Migración, Costos y Gobierno Avanzado"] --> C["Notebook siguiente"]
```


## 1. Framework de Decisión Single vs Multi-Cloud
| Criterio | Pregunta | Indicadores | Peso |
|----------|----------|------------|------|
| Estrategia | ¿Necesitamos resiliencia/regulatorio multi-region? | Países, regulaciones | 0.25 |
| Servicios Diferenciados | ¿Hay features exclusivas críticas? | Benchmark ML, analítica | 0.15 |
| Costos | ¿Podemos optimizar costos con negociación? | % ahorro proyectado | 0.20 |
| Riesgo Vendor Lock-in | ¿Dependencia de APIs propietarias? | Nº integraciones propietarias | 0.15 |
| Capacidad Operativa | ¿Equipo puede sostener complejidad? | Headcount, skills | 0.15 |
| Seguridad/Compliance | ¿Requisitos cross-jurisdiction? | Normativas (GDPR, PCI) | 0.10 |

Decisión favorece multi-cloud si score ponderado > umbral (ej. 7/10).

In [ ]:
# Evaluador simple de decisión multi-cloud
weights = {'estrategia':0.25,'servicios':0.15,'costos':0.20,'lockin':0.15,'operativa':0.15,'compliance':0.10}
scores = {'estrategia':8,'servicios':6,'costos':7,'lockin':5,'operativa':6,'compliance':7}
decision_score = sum(scores[k]*weights[k] for k in weights)
print(f'Score ponderado: {decision_score:.2f}/10')
print('Recomendación: Adoptar multi-cloud' if decision_score>=7 else 'Recomendación: Mantener single-cloud')

## 2. Modelo de Migración por Fases
**Fase 0 (Assessment):** inventario, costos actuales, dependencias, compliance.
**Fase 1 (Pilot/POC):** workload no crítico en segunda nube, métricas iniciales.
**Fase 2 (Dual Run):** replicación selectiva, pruebas de desempeño, políticas unificadas.
**Fase 3 (Controlled Expansion):** dominios adicionales, integración observabilidad y catálogo semántico.
**Fase 4 (Optimization):** balance dinámico, cost arbitrage, spot/reservas, rightsizing.
**Fase 5 (Continuous Improvement):** métricas FinOps y ROI semestral, ADRs revisionados.

### Riesgos Clave
| Riesgo | Impacto | Mitigación |
|--------|---------|-----------|
| Fragmentación de seguridad | Accesos inconsistentes | Centralizar IAM con federación |
| Costos egress inesperados | Aumento TCO | Políticas de data locality y compresión |
| Divergencia de esquemas | Integración difícil | Contratos + ontologías versionadas |
| Operativa sobrecargada | Burnout equipo | Capacitación + automatización IaC |
| ROI inferior al proyectado | Desviación estratégica | Reevaluación trimestral KPIs |

## 3. Simulador de Asignación Dinámica de Workloads
Objetivo: distribuir workloads (ETL, BI, ML, Streaming) para minimizar costo y cumplir latencia.
Criterios simplificados: costo compute y factor latencia por nube. 

In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class Cloud:
    name: str
    compute_cost_per_unit: float
    latency_ms: int

@dataclass
class Workload:
    name: str
    units: int
    max_latency_ms: int

aws = Cloud('AWS', 0.045, 55)
azure = Cloud('Azure', 0.050, 60)
gcp = Cloud('GCP', 0.042, 50)

workloads = [
    Workload('ETL Batch', 1200, 200),
    Workload('BI Interactive', 800, 90),
    Workload('ML Training', 2500, 300),
    Workload('Streaming Events', 1500, 70)
]

def allocate(workloads: List[Workload], clouds: List[Cloud]):
    allocation = {}
    for workload in workloads:
        candidates = [cloud for cloud in clouds if cloud.latency_ms <= workload.max_latency_ms] or clouds
        chosen = min(candidates, key=lambda cloud: cloud.compute_cost_per_unit)
        allocation[workload.name] = {
            'cloud': chosen.name,
            'cost': workload.units * chosen.compute_cost_per_unit,
            'latency_ms': chosen.latency_ms
        }
    return allocation

alloc = allocate(workloads, [aws, azure, gcp])
total_cost = sum(item['cost'] for item in alloc.values())
print('Asignación óptima simplificada:')
for workload, values in alloc.items():
    print(f'{workload}: {values["cloud"]} | costo={values["cost"]:.2f} | latencia={values["latency_ms"]}ms')
print(f'Total estimado: {total_cost:.2f}')


<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- Azure Architecture Center - Data guide: https://learn.microsoft.com/azure/architecture/data-guide/
- Google Cloud Architecture Framework: https://cloud.google.com/architecture/framework
- FinOps Framework: https://www.finops.org/framework/
- NIST AI Risk Management Framework: https://www.nist.gov/itl/ai-risk-management-framework
